**I want to do a time-series regression on the data and analyze whether there was a change in the monthly sentiment of reviews of certain films after September 2001.** Additionally, I can compare monthly sentiments by genre. I have nonfarm payroll numbers and monthly comedy proportion in the market (new varibale) to control with.

#### Dataset: http://localhost:8888/edit/time_series_data.csv?


### Source of Vader Sentiment Analysis: https://www.geeksforgeeks.org/python/python-sentiment-analysis-using-vader/

In [25]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import nltk

def sentiment_scores(sentence):
    sid_obj = SentimentIntensityAnalyzer()
    sentiment_dict = sid_obj.polarity_scores(sentence)
    
    #print(f"Sentiment Scores: {sentiment_dict}")
    #print(f"Negative Sentiment: {sentiment_dict['neg']*100}%")
    #print(f"Neutral Sentiment: {sentiment_dict['neu']*100}%")
    #print(f"Positive Sentiment: {sentiment_dict['pos']*100}%")
    
    #if sentiment_dict['compound'] >= 0.05:
        #print("Overall Sentiment: Positive")
    #elif sentiment_dict['compound'] <= -0.05:
        #print("Overall Sentiment: Negative")
    #else:
        #print("Overall Sentiment: Neutral")

    return sentiment_dict
    
# EXAMPLE
sentiment_scores('It gives a million reason why no one should go.')

{'neg': 0.216, 'neu': 0.784, 'pos': 0.0, 'compound': -0.296}

In [4]:
import requests
import pandas as pd
import numpy as np
import pprint
from bs4 import BeautifulSoup

import os 

from datetime import date
import time
from time import sleep
import selenium #Need this step 
from selenium import webdriver #Need this step 

from selenium.webdriver.common.by import By #Allows for selenium to click things 
from selenium.webdriver.chrome.service import Service #https://stackoverflow.com/questions/64717302/deprecationwarning-executable-path-has-been-deprecated-selenium-python
from selenium.webdriver.support import expected_conditions as EC #Allows for more complex code 
from selenium.webdriver.chrome.options import Options #Allows you to change aspects of the browser

from selenium.common.exceptions import NoSuchElementException

from boxoffice_api import BoxOffice 

In [15]:
chrome_options = Options() 

driver = webdriver.Chrome()
chrome_options.add_argument("--window-size=1900,1000") #pick a size and stick with it, it can change how code intreacts with the website.
driver = webdriver.Chrome(options = chrome_options)

There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache
Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)
There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


In [5]:
titles = []
years = []
guides = []
ratings = []
movie_ids = []

end_year = 2002

https://www.geeksforgeeks.org/software-testing/understanding-no-such-element-exception-in-selenium/

#### All reviews start from 1998, independent of the movie release, and continue up to 2003. When I try to include many movies in my own list, Jupyter stops displaying output. Overall, it takes long for movies with many reviews to load. I have 75 movies, representing each genre: Comedy, Horror, War, Romance, Fantasy. The lists include films released before or in 1998 -- that's when reviews on IMDb start. 



## Comedy

In [8]:
chrome_options = Options() 
driver = webdriver.Chrome()
chrome_options.add_argument("--window-size=1900,1000") #pick a size and stick with it, it can change how code intreacts with the website.
driver = webdriver.Chrome(options = chrome_options)

# -------------------------

comedy_url = "https://www.imdb.com/list/ls4173131935/?ref_=uspf_t_1" 
 
driver.get(comedy_url)
driver.implicitly_wait(10)

buckets = driver.find_elements(By.XPATH, "//li[contains(@class, 'ipc-metadata-list-summary-item')]")

comedy_titles = []
comedy_years = []
comedy_guides = []
comedy_ratings = []
comedy_movie_ids = []

count = 1

for bucket in buckets:
    title_element = bucket.find_element(By.XPATH, ".//h3") # getting the titles
    clean_title = title_element.text.split(". ", 1)[1]  # cleaning tags

    year_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/div/ul/li[1]')
    clean_year = year_element.text
    
    guide_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/div/ul/li[3]')
    clean_guide = guide_element.text

    rating_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/span/div/span/span[1]')
    clean_rating = rating_element.text
    
    comedy_titles.append(clean_title)
    comedy_years.append(clean_year)
    comedy_guides.append(clean_guide)
    comedy_ratings.append(clean_rating)

    # getting movie ids to find the reviews
    id_element = bucket.find_element(By.XPATH, f"/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/ul/div/a").get_attribute('href')
    clean_id = id_element.split('/')[4]
    comedy_movie_ids.append(clean_id)

    count += 1
    
comedy_ids_df = pd.DataFrame(comedy_movie_ids)
comedy_years_df = pd.DataFrame(comedy_years)
comedy_titles_df = pd.DataFrame(comedy_titles)
comedy_guides_df = pd.DataFrame(comedy_guides)
comedy_ratings_df = pd.DataFrame(comedy_ratings)

comedy_df = pd.concat([comedy_ids_df, comedy_years_df, comedy_titles_df, comedy_guides_df, comedy_ratings_df], axis=1)
comedy_df.columns = 'Movie ID', 'Year', 'Title', "Parent's Guide", 'Rating'

print(comedy_df)

# ----------------------------------------------------------------------------------------
# Scraping Reviews
# ----------------------------------------------------------------------------------------

comedy_data = []

driver.implicitly_wait(10)

for movie_id, movie_name in zip(comedy_movie_ids, comedy_titles):
    driver.get(f"https://www.imdb.com/title/{movie_id}/reviews/?sort=submission_date%2Casc&dir=asc")
    
    # we are already at the "reviews" page, sorted by review date in ascending order, e.g., from 1998 onward
    
    count = 1 

    # We keep clicking "Load More" until all reviews up to End_Year are loaded
    # Loop through articles until the last review after clicking "See All" multiple times, which has the year of End_Year - we stop there
    
    while True:
        try: # using NoSuchElementException
            see_all_button = driver.find_element(By.CSS_SELECTOR, ".ipc-see-more__button")
            previous_count = len(driver.find_elements(By.XPATH, "//article"))
            driver.execute_script("arguments[0].click();", see_all_button) # .click() doesn't work, so we .execute_script()

            while len(driver.find_elements(By.XPATH, "//article")) == previous_count: #Waiting until new reviews actually appear
                time.sleep(1) # wait for new reviews to load before clicking again

            # Check the last loaded review's year
            all_reviews = driver.find_elements(By.XPATH, "//article") 
            last_date = all_reviews[-1].find_element(By.XPATH, ".//div[2]/ul/li[2]").text # form: Sep 5, 1998
            last_year = int(last_date.split()[-1].strip()) # check the year

            if last_year > end_year:
                print(f"Loaded up to {last_year} for {movie_name}, stopping clicks")
                break

        except NoSuchElementException:
            print(f"No more 'Load More' button for {movie_name}")
            break

            # 2. Scrape all loaded reviews up to End_Year
    while True:
        try:
            review = driver.find_element(By.XPATH, f"//article[{count}]")
            try:
                review_date = review.find_element(By.XPATH, ".//div[2]/ul/li[2]").text # find the review date first
                year = int(review_date.split()[-1].strip()) # check the year, should be 
                month = review_date.split()[0].strip()

                if year > end_year:
                    print(f"Reached {year} for {movie_name}, moving to next movie")
                    break

                review_text = review.find_element(By.CSS_SELECTOR, ".ipc-overflowText--children").text
                comedy_data.append({
                    'Movie': movie_name,
                    'Date': review_date,
                    'Month, Year': f"{month}, {year}",
                    'Review': review_text
                })
            except NoSuchElementException:
                print(f"Skipping review {count} — missing date or text") # It is most likely SPOILER reviews, doesn't display any text
            count += 1 # adding increment!

        except NoSuchElementException:
            print(f"No more reviews for {movie_name}")
            break

print(comedy_data)

     Movie ID  Year                                              Title  \
0   tt0116282  1996                                              Fargo   
1   tt0109686  1994                                    Dumb and Dumber   
2   tt0109445  1994                                             Clerks   
3   tt0107048  1993                                      Groundhog Day   
4   tt0095705  1988     The Naked Gun: From the Files of Police Squad!   
5   tt0094898  1988                                  Coming to America   
6   tt0091042  1986                           Ferris Bueller's Day Off   
7   tt0087332  1984                                       Ghostbusters   
8   tt0080339  1980                                          Airplane!   
9   tt0080455  1980                                 The Blues Brothers   
10  tt0071853  1975                    Monty Python and the Holy Grail   
11  tt0057012  1964  Dr. Strangelove or: How I Learned to Stop Worr...   
12  tt0053291  1959                   

## Horror

In [10]:
horror_url = "https://www.imdb.com/list/ls4173166845/?ref_=uspf_i_4" 
 
driver.get(horror_url)
driver.implicitly_wait(10)

buckets = driver.find_elements(By.XPATH, "//li[contains(@class, 'ipc-metadata-list-summary-item')]")

horror_titles = []
horror_years = []
horror_guides = []
horror_ratings = []
horror_movie_ids = []

count = 1

for bucket in buckets:
    title_element = bucket.find_element(By.XPATH, ".//h3") # getting the titles
    clean_title = title_element.text.split(". ", 1)[1]  # cleaning tags

    year_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/div/ul/li[1]')
    clean_year = year_element.text
    
    guide_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/div/ul/li[3]')
    clean_guide = guide_element.text

    rating_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/span/div/span/span[1]')
    clean_rating = rating_element.text
    
    horror_titles.append(clean_title)
    horror_years.append(clean_year)
    horror_guides.append(clean_guide)
    horror_ratings.append(clean_rating)

    # getting movie ids to find the reviews
    id_element = bucket.find_element(By.XPATH, f"/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/ul/div/a").get_attribute('href')
    clean_id = id_element.split('/')[4]
    horror_movie_ids.append(clean_id)

    count += 1
    
horror_ids_df = pd.DataFrame(horror_movie_ids)
horror_years_df = pd.DataFrame(horror_years)
horror_titles_df = pd.DataFrame(horror_titles)
horror_guides_df = pd.DataFrame(horror_guides)
horror_ratings_df = pd.DataFrame(horror_ratings)

horror_df = pd.concat([horror_ids_df, horror_years_df, horror_titles_df, horror_guides_df, horror_ratings_df], axis=1)
horror_df.columns = 'Movie ID', 'Year', 'Title', "Parent's Guide", 'Rating'

print(horror_df)

# ----------------------------------------------------------------------------------------
# Scraping Reviews
# ----------------------------------------------------------------------------------------

horror_data = []

driver.implicitly_wait(10)

for movie_id, movie_name in zip(horror_movie_ids, horror_titles):
    driver.get(f"https://www.imdb.com/title/{movie_id}/reviews/?sort=submission_date%2Casc&dir=asc")
    
    # we are already at the "reviews" page, sorted by review date in ascending order, e.g., from 1998 onward
    
    count = 1 

    # We keep clicking "Load More" until all reviews up to End_Year are loaded
    # Loop through articles until the last review after clicking "See All" multiple times, which has the year of End_Year - we stop there
    
    while True:
        try: # using NoSuchElementException
            see_all_button = driver.find_element(By.CSS_SELECTOR, ".ipc-see-more__button")
            previous_count = len(driver.find_elements(By.XPATH, "//article"))
            driver.execute_script("arguments[0].click();", see_all_button) # .click() doesn't work, so we .execute_script()

            while len(driver.find_elements(By.XPATH, "//article")) == previous_count: #Waiting until new reviews actually appear
                time.sleep(1) # wait for new reviews to load before clicking again

            # Check the last loaded review's year
            all_reviews = driver.find_elements(By.XPATH, "//article") 
            last_date = all_reviews[-1].find_element(By.XPATH, ".//div[2]/ul/li[2]").text # form: Sep 5, 1998
            last_year = int(last_date.split()[-1].strip()) # check the year

            if last_year > end_year:
                print(f"Loaded up to {last_year} for {movie_name}, stopping clicks")
                break

        except NoSuchElementException:
            print(f"No more 'Load More' button for {movie_name}")
            break

    # 2. Scrape all loaded reviews up to End_Year
    while True:
        try:
            review = driver.find_element(By.XPATH, f"//article[{count}]")
            try:
                review_date = review.find_element(By.XPATH, ".//div[2]/ul/li[2]").text # find the review date first
                year = int(review_date.split()[-1].strip()) # check the year, should be 
                month = review_date.split()[0].strip()

                if year > end_year:
                    print(f"Reached {year} for {movie_name}, moving to next movie")
                    break

                review_text = review.find_element(By.CSS_SELECTOR, ".ipc-overflowText--children").text
                horror_data.append({
                    'Movie': movie_name,
                    'Date': review_date,
                    'Month, Year': f"{month}, {year}",
                    'Review': review_text
                })
            except NoSuchElementException:
                print(f"Skipping review {count} — missing date or text") # It is most likely SPOILER reviews, doesn't display any text
            count += 1 # adding increment!

        except NoSuchElementException:
            print(f"No more reviews for {movie_name}")
            break

     Movie ID  Year                            Title Parent's Guide Rating
0   tt0119345  1997  I Know What You Did Last Summer              R    5.9
1   tt0117571  1996                           Scream              R    7.4
2   tt0102926  1991         The Silence of the Lambs              R    8.6
3   tt0084787  1982                        The Thing              R    8.2
4   tt0083907  1981                    The Evil Dead          NC-17    7.3
5   tt0081505  1980                      The Shining              R    8.4
6   tt0078748  1979                            Alien              R    8.5
7   tt0077651  1978                        Halloween              R    7.7
8   tt0075005  1976                         The Omen              R    7.5
9   tt0072271  1974     The Texas Chain Saw Massacre              R    7.4
10  tt0070047  1973                     The Exorcist              R    8.1
11  tt0063522  1968                  Rosemary's Baby              R    8.0
12  tt0063350  1968      

## War

In [6]:
chrome_options = Options() 

driver = webdriver.Chrome()
chrome_options.add_argument("--window-size=1900,1000")
driver = webdriver.Chrome(options = chrome_options)

#--------

war_url = "https://www.imdb.com/list/ls4173176957/?ref_=uspf_t_3" 
 
driver.get(war_url)
driver.implicitly_wait(10)

buckets = driver.find_elements(By.XPATH, "//li[contains(@class, 'ipc-metadata-list-summary-item')]")

war_titles = []
war_years = []
war_guides = []
war_ratings = []
war_movie_ids = []

count = 1

for bucket in buckets:
    title_element = bucket.find_element(By.XPATH, ".//h3") # getting the titles
    clean_title = title_element.text.split(". ", 1)[1]  # cleaning tags

    year_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/div/ul/li[1]')
    clean_year = year_element.text
    
    try:
        guide_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/div/ul/li[3]')
        clean_guide = guide_element.text
    except NoSuchElementException:
        clean_guide = "N/A"

    rating_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/span/div/span/span[1]')
    clean_rating = rating_element.text
    
    war_titles.append(clean_title)
    war_years.append(clean_year)
    war_guides.append(clean_guide)
    war_ratings.append(clean_rating)

    # getting movie ids to find the reviews
    id_element = bucket.find_element(By.XPATH, f"/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/ul/div/a").get_attribute('href')
    clean_id = id_element.split('/')[4]
    war_movie_ids.append(clean_id)

    count += 1
    
war_ids_df = pd.DataFrame(war_movie_ids)
war_years_df = pd.DataFrame(war_years)
war_titles_df = pd.DataFrame(war_titles)
war_guides_df = pd.DataFrame(war_guides)
war_ratings_df = pd.DataFrame(war_ratings)

war_df = pd.concat([war_ids_df, war_years_df, war_titles_df, war_guides_df, war_ratings_df], axis=1)
war_df.columns = 'Movie ID', 'Year', 'Title', "Parent's Guide", 'Rating'

print(war_df)

# ----------------------------------------------------------------------------------------
# Scraping Reviews
# ----------------------------------------------------------------------------------------

war_data = []

driver.implicitly_wait(20)

for movie_id, movie_name in zip(war_movie_ids, war_titles):
    driver.get(f"https://www.imdb.com/title/{movie_id}/reviews/?sort=submission_date%2Casc&dir=asc")
    time.sleep(3)
    
    # we are already at the "reviews" page, sorted by review date in ascending order, e.g., from 1998 onward
    
    count = 1 

    # We keep clicking "Load More" until all reviews up to End_Year are loaded
    # Loop through articles until the last review after clicking "See All" multiple times, which has the year of End_Year - we stop there
    
    while True:
        try: # using NoSuchElementException
            see_all_button = driver.find_element(By.CSS_SELECTOR, ".ipc-see-more__button")
            previous_count = len(driver.find_elements(By.XPATH, "//article"))
            driver.execute_script("arguments[0].click();", see_all_button) # .click() doesn't work, so we .execute_script()

            while len(driver.find_elements(By.XPATH, "//article")) == previous_count: #Waiting until new reviews actually appear
                time.sleep(1) # wait for new reviews to load before clicking again

            # Checking the last loaded review's year
            all_reviews = driver.find_elements(By.XPATH, "//article") 
            last_date = all_reviews[-1].find_element(By.XPATH, ".//div[2]/ul/li[2]").text # form: Sep 5, 1998
            last_year = int(last_date.split()[-1].strip()) # check the year

            if last_year > end_year:
                print(f"Loaded up to {last_year} for {movie_name}, stopping clicks")
                break

        except NoSuchElementException:
            print(f"No more 'Load More' button for {movie_name}")
            break

    # 2. Scrape all loaded reviews up to End_Year
    while True:
        try:
            review = driver.find_element(By.XPATH, f"//article[{count}]")
            try:
                review_date = review.find_element(By.XPATH, ".//div[2]/ul/li[2]").text # find the review date first
                year = int(review_date.split()[-1].strip()) # check the year, should be 
                month = review_date.split()[0].strip()

                if year > end_year:
                    print(f"Reached {year} for {movie_name}, moving to next movie")
                    break

                review_text = review.find_element(By.CSS_SELECTOR, ".ipc-overflowText--children").text
                war_data.append({
                    'Movie': movie_name,
                    'Date': review_date,
                    'Month, Year': f"{month}, {year}",
                    'Review': review_text
                })
            except NoSuchElementException:
                print(f"Skipping review {count} — missing date or text") # It is most likely SPOILER reviews, don't display any text
            count += 1 # adding increment!

        except NoSuchElementException:
            print(f"No more reviews for {movie_name}")
            break

print("Finished Scraping!")       

     Movie ID  Year                           Title Parent's Guide Rating
0   tt0120863  1998               The Thin Red Line              R    7.6
1   tt0119173  1997                       G.I. Jane              R    6.0
2   tt0116209  1996             The English Patient              R    7.4
3   tt0097441  1989                           Glory              R    7.8
4   tt0096969  1989      Born on the Fourth of July              R    7.2
5   tt0093058  1987               Full Metal Jacket              R    8.2
6   tt0091763  1986                         Platoon              R    8.1
7   tt0091251  1985                    Come and See      Not Rated    8.3
8   tt0082096  1981                        Das Boot            N/A    8.3
9   tt0078788  1979                  Apocalypse Now              R    8.4
10  tt0077416  1978                 The Deer Hunter              R    8.1
11  tt0066206  1970                          Patton             PG    7.9
12  tt0056172  1962              Lawre

## Romance

In [16]:
romance_url = "https://www.imdb.com/list/ls4173130539/?ref_=uspf_t_5" 
 
driver.get(romance_url)
driver.implicitly_wait(10)

buckets = driver.find_elements(By.XPATH, "//li[contains(@class, 'ipc-metadata-list-summary-item')]")

romance_titles = []
romance_years = []
romance_guides = []
romance_ratings = []
romance_movie_ids = []

count = 1

for bucket in buckets:
    title_element = bucket.find_element(By.XPATH, ".//h3") # getting the titles
    clean_title = title_element.text.split(". ", 1)[1]  # cleaning tags

    year_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/div/ul/li[1]')
    clean_year = year_element.text
    
    guide_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/div/ul/li[3]')
    clean_guide = guide_element.text

    rating_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/span/div/span/span[1]')
    clean_rating = rating_element.text
    
    romance_titles.append(clean_title)
    romance_years.append(clean_year)
    romance_guides.append(clean_guide)
    romance_ratings.append(clean_rating)

    # getting movie ids to find the reviews
    id_element = bucket.find_element(By.XPATH, f"/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/ul/div/a").get_attribute('href')
    clean_id = id_element.split('/')[4]
    romance_movie_ids.append(clean_id)

    count += 1
    
romance_ids_df = pd.DataFrame(romance_movie_ids)
romance_years_df = pd.DataFrame(romance_years)
romance_titles_df = pd.DataFrame(romance_titles)
romance_guides_df = pd.DataFrame(romance_guides)
romance_ratings_df = pd.DataFrame(romance_ratings)

romance_df = pd.concat([romance_ids_df, romance_years_df, romance_titles_df, romance_guides_df, romance_ratings_df], axis=1)
romance_df.columns = 'Movie ID', 'Year', 'Title', "Parent's Guide", 'Rating'

print(romance_df)

# ----------------------------------------------------------------------------------------
# Scraping Reviews
# ----------------------------------------------------------------------------------------

romance_data = []

driver.implicitly_wait(10)

for movie_id, movie_name in zip(romance_movie_ids, romance_titles):
    driver.get(f"https://www.imdb.com/title/{movie_id}/reviews/?sort=submission_date%2Casc&dir=asc")
    
    # we are already at the "reviews" page, sorted by review date in ascending order, e.g., from 1998 onward
    
    count = 1 

    # We keep clicking "Load More" until all reviews up to End_Year are loaded
    # Loop through articles until the last review after clicking "See All" multiple times, which has the year of End_Year - we stop there
    
    while True:
        try: # using NoSuchElementException
            see_all_button = driver.find_element(By.CSS_SELECTOR, ".ipc-see-more__button")
            previous_count = len(driver.find_elements(By.XPATH, "//article"))
            driver.execute_script("arguments[0].click();", see_all_button) # .click() doesn't work, so we .execute_script()

            while len(driver.find_elements(By.XPATH, "//article")) == previous_count: #Waiting until new reviews actually appear
                time.sleep(1) # wait for new reviews to load before clicking again

            # Check the last loaded review's year
            all_reviews = driver.find_elements(By.XPATH, "//article") 
            last_date = all_reviews[-1].find_element(By.XPATH, ".//div[2]/ul/li[2]").text # form: Sep 5, 1998
            last_year = int(last_date.split()[-1].strip()) # check the year

            if last_year > end_year:
                print(f"Loaded up to {last_year} for {movie_name}, stopping clicks")
                break

        except NoSuchElementException:
            print(f"No more 'Load More' button for {movie_name}")
            break

    # 2. Scrape all loaded reviews up to End_Year
    while True:
        try:
            review = driver.find_element(By.XPATH, f"//article[{count}]")
            try:
                review_date = review.find_element(By.XPATH, ".//div[2]/ul/li[2]").text # find the review date first
                year = int(review_date.split()[-1].strip()) # check the year, should be 
                month = review_date.split()[0].strip()

                if year > end_year:
                    print(f"Reached {year} for {movie_name}, moving to next movie")
                    break

                review_text = review.find_element(By.CSS_SELECTOR, ".ipc-overflowText--children").text
                romance_data.append({
                    'Movie': movie_name,
                    'Date': review_date,
                    'Month, Year': f"{month}, {year}",
                    'Review': review_text
                })
            except NoSuchElementException:
                print(f"Skipping review {count} — missing date or text") # It is most likely SPOILER reviews, doesn't display any text
            count += 1 # adding increment!

        except NoSuchElementException:
            print(f"No more reviews for {movie_name}")
            break

print(romance_data)

     Movie ID  Year                        Title Parent's Guide Rating
0   tt0120338  1997                      Titanic          PG-13    8.0
1   tt0119822  1997           As Good as It Gets          PG-13    7.7
2   tt0116695  1996                Jerry Maguire              R    7.3
3   tt0112471  1995               Before Sunrise              R    8.1
4   tt0114388  1995        Sense and Sensibility             PG    7.7
5   tt0109831  1994  Four Weddings and a Funeral              R    7.1
6   tt0108160  1993         Sleepless in Seattle             PG    6.8
7   tt0107048  1993                Groundhog Day             PG    8.0
8   tt0099653  1990                        Ghost          PG-13    7.1
9   tt0100405  1990                 Pretty Woman              R    7.1
10  tt0098635  1989      When Harry Met Sally...              R    7.7
11  tt0075686  1977                   Annie Hall             PG    7.9
12  tt0046250  1953                Roman Holiday       Approved    8.0
13  tt

## Fantasy

In [12]:
fantasy_url = "https://www.imdb.com/list/ls4173166843/?ref_=uspf_t_2" 
 
driver.get(fantasy_url)
driver.implicitly_wait(10)

buckets = driver.find_elements(By.XPATH, "//li[contains(@class, 'ipc-metadata-list-summary-item')]")

fantasy_titles = []
fantasy_years = []
fantasy_guides = []
fantasy_ratings = []
fantasy_movie_ids = []

count = 1

for bucket in buckets:
    title_element = bucket.find_element(By.XPATH, ".//h3") # getting the titles
    clean_title = title_element.text.split(". ", 1)[1]  # cleaning tags

    year_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/div/ul/li[1]')
    clean_year = year_element.text
    
    guide_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/div/ul/li[3]')
    clean_guide = guide_element.text

    rating_element = bucket.find_element(By.XPATH, f'/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/span/div/span/span[1]')
    clean_rating = rating_element.text
    
    fantasy_titles.append(clean_title)
    fantasy_years.append(clean_year)
    fantasy_guides.append(clean_guide)
    fantasy_ratings.append(clean_rating)

    # getting movie ids to find the reviews
    id_element = bucket.find_element(By.XPATH, f"/html/body/div[2]/main/div/section/div/section/div/div[1]/section/div[2]/ul/li[{count}]/div/div/div/div[1]/div[2]/ul/div/a").get_attribute('href')
    clean_id = id_element.split('/')[4]
    fantasy_movie_ids.append(clean_id)

    count += 1
    
fantasy_ids_df = pd.DataFrame(fantasy_movie_ids)
fantasy_years_df = pd.DataFrame(fantasy_years)
fantasy_titles_df = pd.DataFrame(fantasy_titles)
fantasy_guides_df = pd.DataFrame(fantasy_guides)
fantasy_ratings_df = pd.DataFrame(fantasy_ratings)

fantasy_df = pd.concat([fantasy_ids_df, fantasy_years_df, fantasy_titles_df, fantasy_guides_df, fantasy_ratings_df], axis=1)
fantasy_df.columns = 'Movie ID', 'Year', 'Title', "Parent's Guide", 'Rating'

print(fantasy_df)

# ----------------------------------------------------------------------------------------
# Scraping Reviews
# ----------------------------------------------------------------------------------------

fantasy_data = []

driver.implicitly_wait(10)

for movie_id, movie_name in zip(fantasy_movie_ids, fantasy_titles):
    driver.get(f"https://www.imdb.com/title/{movie_id}/reviews/?sort=submission_date%2Casc&dir=asc")
    
    # we are already at the "reviews" page, sorted by review date in ascending order, e.g., from 1998 onward
    
    count = 1 

    # We keep clicking "Load More" until all reviews up to End_Year are loaded
    # Loop through articles until the last review after clicking "See All" multiple times, which has the year of End_Year - we stop there

    while True:
        try: # using NoSuchElementException
            see_all_button = driver.find_element(By.CSS_SELECTOR, ".ipc-see-more__button")
            previous_count = len(driver.find_elements(By.XPATH, "//article"))
            driver.execute_script("arguments[0].click();", see_all_button) # .click() doesn't work, so we .execute_script()

            while len(driver.find_elements(By.XPATH, "//article")) == previous_count: #Waiting until new reviews actually appear
                time.sleep(1) # wait for new reviews to load before clicking again

            # Check the last loaded review's year
            all_reviews = driver.find_elements(By.XPATH, "//article") 
            last_date = all_reviews[-1].find_element(By.XPATH, ".//div[2]/ul/li[2]").text # form: Sep 5, 1998
            last_year = int(last_date.split()[-1].strip()) # check the year

            if last_year > end_year:
                print(f"Loaded up to {last_year} for {movie_name}, stopping clicks")
                break

        except NoSuchElementException:
            print(f"No more 'Load More' button for {movie_name}")
            break

    # 2. Scrape all loaded reviews up to End_Year
    while True:
        try:
            review = driver.find_element(By.XPATH, f"//article[{count}]")
            try:
                review_date = review.find_element(By.XPATH, ".//div[2]/ul/li[2]").text # find the review date first
                year = int(review_date.split()[-1].strip()) # check the year, should be 
                month = review_date.split()[0].strip()

                if year > end_year:
                    print(f"Reached {year} for {movie_name}, moving to next movie")
                    break

                review_text = review.find_element(By.CSS_SELECTOR, ".ipc-overflowText--children").text
                fantasy_data.append({
                    'Movie': movie_name,
                    'Date': review_date,
                    'Month, Year': f"{month}, {year}",
                    'Review': review_text
                })
            except NoSuchElementException:
                print(f"Skipping review {count} — missing date or text") # It is most likely SPOILER reviews, doesn't display any text
            count += 1 # adding increment!

        except NoSuchElementException:
            print(f"No more reviews for {movie_name}")
            break

print(fantasy_data)


     Movie ID  Year                                Title Parent's Guide Rating
0   tt0117008  1996                              Matilda             PG    7.0
1   tt0113497  1995                              Jumanji             PG    7.1
2   tt0102057  1991                                 Hook             PG    6.8
3   tt0099487  1990                  Edward Scissorhands          PG-13    7.8
4   tt0096764  1988   The Adventures of Baron Munchausen             PG    7.1
5   tt0094721  1988                          Beetlejuice             PG    7.4
6   tt0096438  1988              Who Framed Roger Rabbit             PG    7.7
7   tt0093779  1987                   The Princess Bride             PG    8.0
8   tt0091369  1986                            Labyrinth             PG    7.3
9   tt0091203  1986                           Highlander              R    7.0
10  tt0083866  1982           E.T. the Extra-Terrestrial             PG    7.9
11  tt0076759  1977   Star Wars: Episode IV - A New 

In [18]:
comedy_data_df = pd.DataFrame(comedy_data)
comedy_data_df['Genre'] = 'Comedy'

horror_data_df = pd.DataFrame(horror_data)
horror_data_df['Genre'] = 'Horror'

war_data_df = pd.DataFrame(war_data)
war_data_df['Genre'] = 'War'

fantasy_data_df = pd.DataFrame(fantasy_data)
fantasy_data_df['Genre'] = 'Fantasy'

romance_data_df = pd.DataFrame(romance_data)
romance_data_df['Genre'] = 'Romance'

                  Movie          Date Month, Year  \
0                 Fargo  Jul 29, 1998   Jul, 1998   
1                 Fargo  Jul 30, 1998   Jul, 1998   
2                 Fargo  Jul 31, 1998   Jul, 1998   
3                 Fargo   Aug 5, 1998   Aug, 1998   
4                 Fargo   Aug 8, 1998   Aug, 1998   
...                 ...           ...         ...   
13099  The Wizard of Oz   Nov 8, 2002   Nov, 2002   
13100  The Wizard of Oz  Nov 11, 2002   Nov, 2002   
13101  The Wizard of Oz  Nov 24, 2002   Nov, 2002   
13102  The Wizard of Oz  Nov 26, 2002   Nov, 2002   
13103  The Wizard of Oz   Dec 7, 2002   Dec, 2002   

                                                  Review    Genre  
0      This movie does not live up to the hype. It's ...   Comedy  
1      Clearly the best film of 1996 "Fargo" is a bea...   Comedy  
2      I really mean that, too, and I will admit it h...   Comedy  
3      With no major stars or well-known names, the a...   Comedy  
4      This was undoubt

In [48]:
data_df = pd.concat([comedy_data_df, horror_data_df, war_data_df, romance_data_df, fantasy_data_df], ignore_index=True)
print(data_df)

                  Movie          Date Month, Year  \
0                 Fargo  Jul 29, 1998   Jul, 1998   
1                 Fargo  Jul 30, 1998   Jul, 1998   
2                 Fargo  Jul 31, 1998   Jul, 1998   
3                 Fargo   Aug 5, 1998   Aug, 1998   
4                 Fargo   Aug 8, 1998   Aug, 1998   
...                 ...           ...         ...   
13099  The Wizard of Oz   Nov 8, 2002   Nov, 2002   
13100  The Wizard of Oz  Nov 11, 2002   Nov, 2002   
13101  The Wizard of Oz  Nov 24, 2002   Nov, 2002   
13102  The Wizard of Oz  Nov 26, 2002   Nov, 2002   
13103  The Wizard of Oz   Dec 7, 2002   Dec, 2002   

                                                  Review    Genre  
0      This movie does not live up to the hype. It's ...   Comedy  
1      Clearly the best film of 1996 "Fargo" is a bea...   Comedy  
2      I really mean that, too, and I will admit it h...   Comedy  
3      With no major stars or well-known names, the a...   Comedy  
4      This was undoubt

In [49]:
# How many reviews each movie has up to a certain date

for movie in data_df['Movie'].unique():
    review_number = len(data_df[data_df['Movie']== movie])
    print(f"{movie} has", review_number, "reviews.")

Fargo has 251 reviews.
Dumb and Dumber has 152 reviews.
Clerks has 280 reviews.
Groundhog Day has 362 reviews.
The Naked Gun: From the Files of Police Squad! has 53 reviews.
Coming to America has 37 reviews.
Ferris Bueller's Day Off has 171 reviews.
Ghostbusters has 142 reviews.
Airplane! has 115 reviews.
The Blues Brothers has 88 reviews.
Monty Python and the Holy Grail has 255 reviews.
Dr. Strangelove or: How I Learned to Stop Worrying and Love the Bomb has 253 reviews.
Some Like It Hot has 79 reviews.
Modern Times has 38 reviews.
Duck Soup has 67 reviews.
I Know What You Did Last Summer has 221 reviews.
Scream has 386 reviews.
The Silence of the Lambs has 258 reviews.
The Thing has 239 reviews.
The Evil Dead has 235 reviews.
The Shining has 338 reviews.
Alien has 245 reviews.
Halloween has 357 reviews.
The Omen has 71 reviews.
The Texas Chain Saw Massacre has 254 reviews.
The Exorcist has 382 reviews.
Rosemary's Baby has 87 reviews.
Night of the Living Dead has 169 reviews.
The Bird

In [50]:
# Create a list of the "compound" sentiment scores
sentiment_score = [sentiment_scores(review)['compound'] for review in data_df['Review']]
print(sentiment_score)

# Adding the new column of sentiment to our main dataframe
data_df['Separate_Sentiment_Score'] = sentiment_score
sentiment = [sentiment_scores(review)['compound'] for review in data_df['Review']]

# Adding the "Sentiment Overview" column to our dataframe
data_df['Sentiment'] = [
    'Positive' if score > 0.05 
    else 'Negative' if score < -0.05 
    else 'Neutral' 
    for score in data_df['Separate_Sentiment_Score']
]

# Now, we calculate monthly compound sentiment

data_df['Date'] = pd.to_datetime(data_df['Date']) # converting to datetime

data_df.sort_values(by = 'Date') # sorting by review date

# getting the average sentiment score for each month, grouping by dt.to_period('M')
data_df['Monthly_Sentiment_Score'] = data_df.groupby(data_df['Date'].dt.to_period('M'))['Separate_Sentiment_Score'].transform('mean')

[-0.3153, 0.9758, 0.9692, 0.7258, 0.7964, -0.5563, -0.0897, -0.0196, 0.9383, 0.5859, 0.9702, 0.9881, 0.9908, 0.7707, 0.5789, 0.9892, 0.0415, -0.5624, 0.8421, 0.9816, 0.6932, 0.9916, 0.9609, -0.3574, 0.8481, 0.9856, 0.9886, 0.9193, -0.1027, 0.9561, 0.9534, 0.9262, 0.9908, 0.9883, 0.7302, -0.6985, 0.9501, -0.9096, 0.5408, -0.8796, 0.8889, 0.841, -0.747, 0.722, 0.908, 0.6901, 0.3893, 0.9835, 0.9878, -0.5974, 0.9318, 0.996, 0.7003, 0.9923, -0.3823, 0.5692, 0.9758, 0.6775, -0.9542, 0.628, 0.3079, 0.9788, -0.6766, -0.7783, 0.9158, 0.9701, 0.9612, 0.9876, -0.9817, -0.0344, 0.7237, 0.8206, 0.5751, 0.9742, 0.8848, 0.9616, 0.9984, 0.8461, 0.3744, 0.5859, 0.9255, 0.8888, 0.6542, 0.9947, 0.9601, 0.9908, 0.8355, 0.9333, 0.9118, 0.9735, 0.907, 0.9781, 0.9251, 0.9054, 0.984, 0.9714, 0.5561, 0.9958, 0.9656, -0.0945, 0.9929, -0.8666, 0.6393, 0.9929, 0.9287, 0.3661, 0.948, -0.9588, 0.9858, 0.9871, 0.1796, 0.9808, 0.9754, 0.9879, 0.9825, 0.942, 0.7059, -0.0981, 0.9852, 0.9202, 0.8934, 0.9693, 0.9463, 0.9

In [72]:
print(data_df.head(5))

   Movie       Date Month, Year  \
0  Fargo 1998-07-29   Jul, 1998   
1  Fargo 1998-07-30   Jul, 1998   
2  Fargo 1998-07-31   Jul, 1998   
3  Fargo 1998-08-05   Aug, 1998   
4  Fargo 1998-08-08   Aug, 1998   

                                              Review   Genre  \
0  This movie does not live up to the hype. It's ...  Comedy   
1  Clearly the best film of 1996 "Fargo" is a bea...  Comedy   
2  I really mean that, too, and I will admit it h...  Comedy   
3  With no major stars or well-known names, the a...  Comedy   
4  This was undoubtedly the best film of 1996. Th...  Comedy   

   Separate_Sentiment_Score Sentiment  Monthly_Sentiment_Score  
0                   -0.3153  Negative                 0.725789  
1                    0.9758  Positive                 0.725789  
2                    0.9692  Positive                 0.725789  
3                    0.7258  Positive                 0.477847  
4                    0.7964  Positive                 0.477847  


In [91]:
print(data_df[(data_df['Genre'] == 'War') & (data_df['Month, Year'] == 'Jul, 1998')].head(5))
# For War there are no reviews in July, reviews start in August
print(data_df[(data_df['Genre'] == 'War') & (data_df['Month, Year'] == 'Aug, 1998')].head(3))

Empty DataFrame
Columns: [Movie, Date, Month, Year, Review, Genre, Separate_Sentiment_Score, Sentiment, Monthly_Sentiment_Score]
Index: []
                    Movie       Date Month, Year  \
6736            G.I. Jane 1998-08-16   Aug, 1998   
6790  The English Patient 1998-08-06   Aug, 1998   
6791  The English Patient 1998-08-07   Aug, 1998   

                                                 Review Genre  \
6736  It's disconcerting to see Demi Moore in such a...   War   
6790  It is definitely my favorite movie of all time...   War   
6791  There is pain in this movie, but there is happ...   War   

      Separate_Sentiment_Score Sentiment  Monthly_Sentiment_Score  
6736                    0.9778  Positive                 0.477847  
6790                    0.9462  Positive                 0.477847  
6791                    0.9520  Positive                 0.477847  


## Monthly Sentiment Dataset

In [95]:
# Monthly Sentiment

monthly_sentiment_df = data_df[['Month, Year', 'Monthly_Sentiment_Score']] # separating the dataframe for Month vs Sentiment Score
monthly_sentiments_df = monthly_sentiment_df.groupby('Month, Year')['Monthly_Sentiment_Score'].mean().reset_index() # get one score per month

# Convert to datetime, sort, then format back to string

monthly_sentiments_df['Month, Year'] = pd.to_datetime(monthly_sentiments_df['Month, Year'], format='%b, %Y') # convert to datetime format
monthly_sentiments_df = monthly_sentiments_df.sort_values('Month, Year').reset_index(drop = True) # sort months-years
monthly_sentiments_df['Month, Year'] = monthly_sentiments_df['Month, Year'].dt.strftime('%b, %Y')  # convert back to original string format

print(monthly_sentiments_df.head(5))

  Month, Year  Monthly_Sentiment_Score
0   Jul, 1998                 0.725789
1   Aug, 1998                 0.477847
2   Sep, 1998                 0.474915
3   Oct, 1998                 0.381850
4   Nov, 1998                 0.411975


In [96]:
# We Include Monthly Scores by Genre
# Comedy
comedy_monthly = (
    data_df[data_df['Genre'] == 'Comedy']
    .groupby('Month, Year')['Separate_Sentiment_Score']
    .mean()
    .rename('Comedy_Monthly_Sentiment_Score')
)

monthly_sentiments_df = monthly_sentiments_df.join(comedy_monthly, on='Month, Year')

# Horror
horror_monthly = (
    data_df[data_df['Genre'] == 'Horror']
    .groupby('Month, Year')['Separate_Sentiment_Score']
    .mean()
    .rename('Horror_Monthly_Sentiment_Score')
)

monthly_sentiments_df = monthly_sentiments_df.join(horror_monthly, on='Month, Year')

# War
war_monthly = (
    data_df[data_df['Genre'] == 'War']
    .groupby('Month, Year')['Separate_Sentiment_Score']
    .mean()
    .rename('War_Monthly_Sentiment_Score')
)

monthly_sentiments_df = monthly_sentiments_df.join(war_monthly, on='Month, Year')

# Romance
romance_monthly = (
    data_df[data_df['Genre'] == 'Romance']
    .groupby('Month, Year')['Separate_Sentiment_Score']
    .mean()
    .rename('Romance_Monthly_Sentiment_Score')
)

monthly_sentiments_df = monthly_sentiments_df.join(romance_monthly, on='Month, Year')

# Fantasy
fantasy_monthly = (
    data_df[data_df['Genre'] == 'Fantasy']
    .groupby('Month, Year')['Separate_Sentiment_Score']
    .mean()
    .rename('Fantasy_Monthly_Sentiment_Score')
)

monthly_sentiments_df = monthly_sentiments_df.join(fantasy_monthly, on='Month, Year')

print(monthly_sentiments_df.head(7))

  Month, Year  Monthly_Sentiment_Score  Comedy_Monthly_Sentiment_Score  \
0   Jul, 1998                 0.725789                        0.653467   
1   Aug, 1998                 0.477847                        0.508308   
2   Sep, 1998                 0.474915                        0.656311   
3   Oct, 1998                 0.381850                        0.717947   
4   Nov, 1998                 0.411975                        0.623088   
5   Dec, 1998                 0.393935                        0.791300   
6   Jan, 1999                 0.328322                        0.652346   

   Horror_Monthly_Sentiment_Score  War_Monthly_Sentiment_Score  \
0                        0.815233                          NaN   
1                        0.111763                     0.533457   
2                        0.008666                     0.769713   
3                        0.026919                     0.307859   
4                        0.007610                     0.502912   
5          

### Gathering Box Office API to Create a New Variable: 
We get monthly box office data between 1998 and 2002 - 4 years of data.

https://pypi.org/project/boxoffice-api

In [99]:
# DON'T RUN IT!!!

box_office = BoxOffice(api_key="5d968797") 
monthly_box_office = []

for year in range(1998, 2003): # not includign 2003
    for month in range(1, 13): # not including 13
        monthly_data = box_office.get_monthly(year = year, month = month)
        monthly_box_office.append(monthly_data)
        
        time.sleep(1)
        
print(monthly_box_office)

# DON'T RUN IT!!!

[[{'Rank': '1', 'Release': 'Titanic', 'Gross': '$188,215,666', 'Theaters': '3,265', 'Total Gross': '$600,683,057', 'Release Date': 'Dec 19', 'Distributor': 'Paramount Pictures', 'Response': 'False', 'Error': 'Movie not found!'}, {'Rank': '2', 'Release': 'As Good as It Gets', 'Gross': '$59,523,768', 'Theaters': '1,837', 'Total Gross': '$148,478,011', 'Release Date': 'Dec 23', 'Distributor': 'Sony Pictures Releasing', 'Response': 'False', 'Error': 'Movie not found!'}, {'Rank': '3', 'Release': 'Good Will Hunting', 'Gross': '$49,409,331', 'Theaters': '2,203', 'Total Gross': '$138,433,435', 'Release Date': 'Dec 5', 'Distributor': 'Miramax', 'Response': 'False', 'Error': 'Movie not found!'}, {'Rank': '4', 'Release': 'Tomorrow Never Dies', 'Gross': '$42,773,740', 'Theaters': '2,807', 'Total Gross': '$125,304,276', 'Release Date': 'Dec 19', 'Distributor': 'Metro-Goldwyn-Mayer (MGM)', 'Response': 'False', 'Error': 'Movie not found!'}, {'Rank': '5', 'Release': 'Mousehunt', 'Gross': '$27,515,643'

In [112]:
target_movies = list(set([movie['Release'] for month in monthly_box_office for movie in month])) # set removes duplicates
# print (target_movies)
# Comedy movies in our box office leads

In [102]:
# This is so that IMDb allows us to perform stuff and doesn't block us (as hinted by Jupyter assistant)
import random
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument("--window-size=1900,1000")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option("useAutomationExtension", False)
chrome_options.add_argument("--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=chrome_options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")
# --------------------------------------------------------------------------------------------------

comedy_url = "https://www.imdb.com/search/title/?title_type=feature&release_date=1998-01-01,2005-12-31&genres=comedy&moviemeter=,50000"
# we look for comedies released between 1998 and 2005 and are in the top 50,000 popular films on IMDb
driver.get(comedy_url)
time.sleep(3)

while True:
    try:
        load_more_btn = driver.find_element(By.CSS_SELECTOR, ".ipc-see-more__button") # Now trying to find the "50 more" button
        driver.execute_script("arguments[0].click();", load_more_btn) # Clicking it using JavaScript (to avoid "Intercepted" errors)
        time.sleep(3) # wait
        
    except NoSuchElementException:
        print("All Movies Are Loaded")
        break

comedies_on_page = driver.find_elements(By.XPATH, "/html/body/div[2]/main/div[2]/div[3]/section/section/div/section/section/div[2]/div/section/div[2]/div[2]/ul/li")

found = False

top_comedies = []

for item in comedies_on_page:
    card_text = item.text #
    
    for target_movie in target_movies:
        if target_movie in card_text:
            print(f"{target_movie} is a Box Office Hit.")
            top_comedies.append(target_movie)
        
            found = True
            break

All Movies Are Loaded
The Truman Show is a Box Office Hit.
Rat is a Box Office Hit.
The Big Lebowski is a Box Office Hit.
Scary Movie is a Box Office Hit.
10 Things I Hate About You is a Box Office Hit.
Rat is a Box Office Hit.
Amélie is a Box Office Hit.
Rat is a Box Office Hit.
Wet Hot American Summer is a Box Office Hit.
Rat is a Box Office Hit.
American Pie is a Box Office Hit.
Toy Story 2 is a Box Office Hit.
Shrek is a Box Office Hit.
Rat is a Box Office Hit.
Rat is a Box Office Hit.
Rat is a Box Office Hit.
O is a Box Office Hit.
Rat is a Box Office Hit.
Rat is a Box Office Hit.
Monsters, Inc. is a Box Office Hit.
Rat is a Box Office Hit.
Go is a Box Office Hit.
Rat is a Box Office Hit.
Rat is a Box Office Hit.
Scooby-Doo is a Box Office Hit.
Almost Famous is a Box Office Hit.
Rat is a Box Office Hit.
O is a Box Office Hit.
O is a Box Office Hit.
Lock, Stock and Two Smoking Barrels is a Box Office Hit.
Rat is a Box Office Hit.
Ice Age is a Box Office Hit.
The Royal Tenenbaums is

In [113]:
unique_comedies = set(top_comedies)
print(unique_comedies)

{"There's Something About Mary", 'Scooby-Doo', 'Spy Kids', 'American Movie', 'Barbershop', 'Punch-Drunk Love', 'Jungle', 'Babe: Pig in the City', 'Play It to the Bone', 'Meet the Deedles', 'Wasabi', 'Crossroads', 'Almost Famous', 'Dr. Dolittle 2', 'Breakfast of Champions', 'Ready to Rumble', 'Bowfinger', 'Hanging Up', 'The Powerpuff Girls Movie', 'Lovely & Amazing', 'Traffic', 'Bamboozled', 'The Adventures of Pluto Nash', 'Storytelling', 'Harvard Man', 'Mean Machine', 'Zero Effect', 'Twice Upon a Yesterday', 'Ghost World', 'Baby Geniuses', 'Max', 'Iris', 'Romance', '3 Strikes', 'Fear and Loathing in Las Vegas', 'Very Bad Things', 'Lucky Numbers', 'Someone Like You', 'O', "But I'm a Cheerleader", 'Inspector Gadget', 'CQ', 'Trick', 'Serendipity', 'Son of the Bride', 'Happiness', 'See Spot Run', "Cookie's Fortune", 'Rushmore', 'Fidel', 'Welcome to Collinwood', 'Snow Dogs', 'Lake Placid', 'Tadpole', 'MVP: Most Valuable Primate', 'Gloria', 'The Little Vampire', 'Holy Smoke', 'Edtv', 'Saving

#### We're going to count how many comedy movies ranked each month and divide by the total number of ranks to reflect the proportion of happiness in the market

In [138]:
# API data is the List of Lists
# monthly_box_office = [[{'Release': 'Titanic'...}], [{'Release': 'The Wedding Singer'...}], ...] -- goes like this

data_for_df = []
timeline_labels = []

# This builds a list like: ["Jan, 1998", "Feb, 1998", "Mar, 1998" ... "Dec, 2002"]
for year in range(1998, end_year+1): 
    for month in months:
        timeline_labels.append(f"{month}, {year}")

for index, month_data in enumerate(monthly_box_office):
    current_month_label = timeline_labels[index]
    
    last_movie = month_data[-1]
    max_rank = int(last_movie['Rank']) # identifiying the maximum rank of box offcie hits
    #ranks.append(max_rank)
    
    comedy_count = 0 # counting top-ranked comedies in each month
    for movie in month_data:
        if movie['Release'] in unique_comedies:
            comedy_count += 1
    
    data_for_df.append([current_month_label, comedy_count, max_rank])
    
comedy_pop_df = pd.DataFrame(data_for_df, columns=['Month, Year', 'Comedy_Count', 'Maximum_Rank'])

print(comedy_pop_df)

   Month, Year  Comedy_Count  Maximum_Rank
0    Jan, 1998             0            45
1    Feb, 1998             3            46
2    Mar, 1998             6            54
3    Apr, 1998             8            61
4    May, 1998             8            54
5    Jun, 1998            10            46
6    Jul, 1998            10            50
7    Aug, 1998            14            46
8    Sep, 1998            13            54
9    Oct, 1998            10            54
10   Nov, 1998            18            50
11   Dec, 1998            22            62
12   Jan, 1999            25            85
13   Feb, 1999            26            89
14   Mar, 1999            22            94
15   Apr, 1999            28           105
16   May, 1999            28           104
17   Jun, 1999            29           100
18   Jul, 1999            26            97
19   Aug, 1999            25            98
20   Sep, 1999            21           101
21   Oct, 1999            27           110
22   Nov, 1

We join two dataframes: Monthly Sentiment from the data dataframe and the Comedy Count dataframe.

### Let's calculate the new variable:

In [139]:
# Convert the 'Month & Year' column to datetime format first, then perform the comparison

comedy_pop_df['Month, Year'] = pd.to_datetime(comedy_pop_df['Month, Year']) # convert it to a certain datetime
comedy_pop_df = comedy_pop_df[comedy_pop_df['Month, Year'] > pd.to_datetime('Jun 1998')].reset_index(drop = True) # Filter
comedy_pop_df['Month, Year'] = comedy_pop_df['Month, Year'].dt.strftime('%b, %Y') # get it back to previos date format

comedy_pop_df['Comedy_Proportion'] =  comedy_pop_df['Comedy_Count']/comedy_pop_df['Maximum_Rank'] # calculate the new variable

print(comedy_pop_df)

   Month, Year  Comedy_Count  Maximum_Rank  Comedy_Proportion
0    Jul, 1998            10            50           0.200000
1    Aug, 1998            14            46           0.304348
2    Sep, 1998            13            54           0.240741
3    Oct, 1998            10            54           0.185185
4    Nov, 1998            18            50           0.360000
5    Dec, 1998            22            62           0.354839
6    Jan, 1999            25            85           0.294118
7    Feb, 1999            26            89           0.292135
8    Mar, 1999            22            94           0.234043
9    Apr, 1999            28           105           0.266667
10   May, 1999            28           104           0.269231
11   Jun, 1999            29           100           0.290000
12   Jul, 1999            26            97           0.268041
13   Aug, 1999            25            98           0.255102
14   Sep, 1999            21           101           0.207921
15   Oct

/var/folders/xh/nk0kvhgx4xs4vzfmwkft2hpm0000gn/T/ipykernel_56395/1373709299.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  comedy_pop_df['Month, Year'] = pd.to_datetime(comedy_pop_df['Month, Year']) # convert it to a certain datetime


## Time Series Dataset

In [179]:
time_series_df = pd.merge(comedy_pop_df, monthly_sentiments_df, on = 'Month, Year', how = 'inner') # inner joining the comedy and sentiments dfs
time_series_df = time_series_df.drop(columns = (['Comedy_Count', 'Maximum_Rank'])) # dropping all comedy columns but the new variable

print(time_series_df.head(5))

  Month, Year  Comedy_Proportion  Monthly_Sentiment_Score  \
0   Jul, 1998           0.200000                 0.725789   
1   Aug, 1998           0.304348                 0.477847   
2   Sep, 1998           0.240741                 0.474915   
3   Oct, 1998           0.185185                 0.381850   
4   Nov, 1998           0.360000                 0.411975   

   Comedy_Monthly_Sentiment_Score  Horror_Monthly_Sentiment_Score  \
0                        0.653467                        0.815233   
1                        0.508308                        0.111763   
2                        0.656311                        0.008666   
3                        0.717947                        0.026919   
4                        0.623088                        0.007610   

   War_Monthly_Sentiment_Score  Romance_Monthly_Sentiment_Score  \
0                          NaN                         0.733185   
1                     0.533457                         0.683165   
2                

In [180]:
time_series_df['Month, Year'] = pd.to_datetime(time_series_df['Month, Year'])

time_series_df['9/11'] = np.where(
    time_series_df['Month, Year'] < pd.to_datetime('2001-09-01'),
    0,  # before Sep 2001
    1)   # Sep 2001 and after

time_series_df['Month, Year'] = time_series_df['Month, Year'].dt.strftime('%b, %Y') # get it back to previous date format

print(time_series_df.head(5))

  Month, Year  Comedy_Proportion  Monthly_Sentiment_Score  \
0   Jul, 1998           0.200000                 0.725789   
1   Aug, 1998           0.304348                 0.477847   
2   Sep, 1998           0.240741                 0.474915   
3   Oct, 1998           0.185185                 0.381850   
4   Nov, 1998           0.360000                 0.411975   

   Comedy_Monthly_Sentiment_Score  Horror_Monthly_Sentiment_Score  \
0                        0.653467                        0.815233   
1                        0.508308                        0.111763   
2                        0.656311                        0.008666   
3                        0.717947                        0.026919   
4                        0.623088                        0.007610   

   War_Monthly_Sentiment_Score  Romance_Monthly_Sentiment_Score  \
0                          NaN                         0.733185   
1                     0.533457                         0.683165   
2                

/var/folders/xh/nk0kvhgx4xs4vzfmwkft2hpm0000gn/T/ipykernel_56395/1932107580.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  time_series_df['Month, Year'] = pd.to_datetime(time_series_df['Month, Year'])


#### API from St.Louis Fed: https://fred.stlouisfed.org/docs/api/fred/series_observations.html#examples

In [151]:
pip install fredapi pandas

Note: you may need to restart the kernel to use updated packages.


In [157]:
from fredapi import Fred

api_key='1b687f28e5349ffe291339b9f8591d8c' 
series_id = 'PAYEMS'           # Using NonFarm Payrolls as a monthly proxy
start_date = '1998-07-01'
end_date = '2002-12-31'

url = (
    f"https://api.stlouisfed.org/fred/series/observations?"
    f"series_id={series_id}&"
    f"api_key={api_key}&"
    f"observation_start={start_date}&"
    f"observation_end={end_date}&"
    f"frequency=m&"
    f"file_type=json"
)

response = requests.get(url)
data = response.json()

print(data)

{'realtime_start': '2026-04-11', 'realtime_end': '2026-04-11', 'observation_start': '1998-07-01', 'observation_end': '2002-12-31', 'units': 'lin', 'output_type': 1, 'file_type': 'json', 'order_by': 'observation_date', 'sort_order': 'asc', 'count': 54, 'offset': 0, 'limit': 100000, 'observations': [{'realtime_start': '2026-04-11', 'realtime_end': '2026-04-11', 'date': '1998-07-01', 'value': '126212'}, {'realtime_start': '2026-04-11', 'realtime_end': '2026-04-11', 'date': '1998-08-01', 'value': '126546'}, {'realtime_start': '2026-04-11', 'realtime_end': '2026-04-11', 'date': '1998-09-01', 'value': '126744'}, {'realtime_start': '2026-04-11', 'realtime_end': '2026-04-11', 'date': '1998-10-01', 'value': '126949'}, {'realtime_start': '2026-04-11', 'realtime_end': '2026-04-11', 'date': '1998-11-01', 'value': '127222'}, {'realtime_start': '2026-04-11', 'realtime_end': '2026-04-11', 'date': '1998-12-01', 'value': '127595'}, {'realtime_start': '2026-04-11', 'realtime_end': '2026-04-11', 'date': 

In [172]:
observations = data['observations']

dates = [obs['date'] for obs in observations]
payroll_values = [int(obs['value']) for obs in observations] # Extract dates and convert values to integers

print(dates)
print(payroll_values)

payroll_df = pd.DataFrame(list(zip(dates, payroll_values)), columns=['Month, Year', 'Payroll'])
payroll_df['Month, Year'] = time_series_df['Month, Year']
print(payroll_df)

['1998-07-01', '1998-08-01', '1998-09-01', '1998-10-01', '1998-11-01', '1998-12-01', '1999-01-01', '1999-02-01', '1999-03-01', '1999-04-01', '1999-05-01', '1999-06-01', '1999-07-01', '1999-08-01', '1999-09-01', '1999-10-01', '1999-11-01', '1999-12-01', '2000-01-01', '2000-02-01', '2000-03-01', '2000-04-01', '2000-05-01', '2000-06-01', '2000-07-01', '2000-08-01', '2000-09-01', '2000-10-01', '2000-11-01', '2000-12-01', '2001-01-01', '2001-02-01', '2001-03-01', '2001-04-01', '2001-05-01', '2001-06-01', '2001-07-01', '2001-08-01', '2001-09-01', '2001-10-01', '2001-11-01', '2001-12-01', '2002-01-01', '2002-02-01', '2002-03-01', '2002-04-01', '2002-05-01', '2002-06-01', '2002-07-01', '2002-08-01', '2002-09-01', '2002-10-01', '2002-11-01', '2002-12-01']
[126212, 126546, 126744, 126949, 127222, 127595, 127699, 128117, 128234, 128595, 128813, 129084, 129422, 129572, 129771, 130178, 130466, 130783, 131011, 131120, 131604, 131884, 132105, 132063, 132228, 132234, 132337, 132352, 132556, 132716, 13

In [177]:
time_series_df = pd.merge(time_series_df, payroll_df, on = 'Month, Year', how = 'left')

#time_series_df = time_series_df.drop(columns=['Payroll_x', 'Payroll_y','Payroll_new'], errors='ignore')
print(time_series_df)

   Month, Year  Comedy_Proportion  Monthly_Sentiment_Score  \
0    Jul, 1998           0.200000                 0.725789   
1    Aug, 1998           0.304348                 0.477847   
2    Sep, 1998           0.240741                 0.474915   
3    Oct, 1998           0.185185                 0.381850   
4    Nov, 1998           0.360000                 0.411975   
5    Dec, 1998           0.354839                 0.393935   
6    Jan, 1999           0.294118                 0.328322   
7    Feb, 1999           0.292135                 0.385467   
8    Mar, 1999           0.234043                 0.337991   
9    Apr, 1999           0.266667                 0.383760   
10   May, 1999           0.269231                 0.444659   
11   Jun, 1999           0.290000                 0.392103   
12   Jul, 1999           0.268041                 0.378658   
13   Aug, 1999           0.255102                 0.460306   
14   Sep, 1999           0.207921                 0.443523   
15   Oct

In [178]:
time_series_df.to_csv('time_series_data.csv', index=False)

In [182]:
driver.quit()

Prof K
Well, this is a novel. You ned to document - what line is each thing doing?

To simplify, I suggest you get all of the titles for each genre in the first part. THEN you run the scrapping data for the entire list for every movie name you want to scrape in the movie id review website.

The below code IS SO COMLICATED and does not need to be. you can write this in like 4 lines of code.

SIMPLIFY SIMPLIFY SIMPLIFY

data_for_df = []
timeline_labels = []

#This builds a list like: ["Jan, 1998", "Feb, 1998", "Mar, 1998" ... "Dec, 2002"]
for year in range(1998, end_year+1): 
    for month in months:
        timeline_labels.append(f"{month}, {year}")

for index, month_data in enumerate(monthly_box_office):
    current_month_label = timeline_labels[index]
    
    last_movie = month_data[-1]
    max_rank = int(last_movie['Rank']) # identifiying the maximum rank of box offcie hits
    #ranks.append(max_rank)
    
    comedy_count = 0 # counting top-ranked comedies in each month
    for movie in month_data:
        if movie['Release'] in unique_comedies:
            comedy_count += 1
    
    data_for_df.append([current_month_label, comedy_count, max_rank])
    
comedy_pop_df = pd.DataFrame(data_for_df, columns=['Month, Year', 'Comedy_Count', 'Maximum_Rank'])

print(comedy_pop_df)

For the final project:
1. more documentation
2. More efficiency and less linces of code.
3. very clever sentiment index/new variable. talk abotu this in your final project for a bit. 

extra points for creativity, but careful on the thousands of lines and the try/except from chatgpt.

grade: 87


